In [ ]:
!pip install mlflow -q
!pip install --upgrade transformers
!pip uninstall -y torchao


In [ ]:
import json, random, torch, numpy as np
from datetime import datetime
from sklearn.metrics import f1_score, accuracy_score, recall_score
import mlflow
from datasets import Dataset, load_dataset, concatenate_datasets, Value
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer, pipeline, TrainerCallback)
from transformers import logging as hf_logging
from huggingface_hub import login, HfApi, create_repo, hf_hub_download, repo_exists
from peft import LoraConfig, get_peft_model, TaskType
import os
hf_logging.set_verbosity_error()

def get_secret(label, fallback_path):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(label)
    except Exception:
        pass

    if os.path.exists(fallback_path):
        with open(fallback_path) as f:
            return json.load(f)[label]

    raise ValueError(f"Secret '{label}' not found")

HF_USER =           get_secret("HF_USER", "/kaggle/input/secrets/secrets.json")
HF_TOKEN =          get_secret("HF_TOKEN", "/kaggle/input/secrets/secrets.json")
SPACE_URL =         get_secret("SPACE_URL", "/kaggle/input/secrets/secrets.json")
SYNTH_DATA_REPO =   HF_USER + "/" + get_secret("SYNTH_DATA_REPO","/kaggle/input/secrets/secrets.json")
MODEL_REPO   =      HF_USER + "/" + get_secret("MODEL_REPO", "/kaggle/input/secrets/secrets.json")
METRICS_REPO =      HF_USER + "/" + get_secret("METRICS_REPO", "/kaggle/input/secrets/secrets.json")
SPACE_NAME =        get_secret("SPACE_NAME", "/kaggle/input/secrets/secrets.json")

login(token=HF_TOKEN)

COMPANY       = "Anthropic"
COMPANY_DESC  = "Anthropic is an AI research and safety company that develops large language models, most notably Claude"
BASE_MODEL    = "cardiffnlp/twitter-roberta-base-sentiment-latest"
N_PER_CLASS   = 100
LABEL_MAP     = {"negative": 0, "neutral": 1, "positive": 2}

ASPECT = COMPANY # Aspect Based Sentiment Analysis (ABSA) -> will relate the specific aspect with its sentiment in the text
GENERIC_ASPECT = "this topic"

MODEL_EXISTS = HF_USER and MODEL_REPO and repo_exists(MODEL_REPO)


mlflow.set_tracking_uri(f"{SPACE_URL}/mlflow")
create_repo(METRICS_REPO, repo_type="dataset", exist_ok=True)


RepoUrl('https://huggingface.co/datasets/divde/sentiment-training-metrics', endpoint='https://huggingface.co', repo_type='dataset', repo_id='divde/sentiment-training-metrics')

In [ ]:
try:
    COLLECTED_DATA = hf_hub_download(repo_id=f"{HF_USER}/sentiment_posts", filename="new_posts.json", repo_type="dataset")
    collected_posts = json.load(open(COLLECTED_DATA))
    collected_text = [post["text"] for post in collected_posts]
except Exception:
    print("No data collected in database, using public database")
    fallback_data = load_dataset("tweet_eval", "sentiment", split="train")
    collected_text = fallback_data["text"]


In [ ]:
generator = pipeline(
    "text-generation",
    model="google/gemma-4-e2b-it",
    dtype=torch.float16,
    device=0,
    max_length=None,
    max_new_tokens=80,
    do_sample=True,
    temperature=1.2,
    top_p= 0.92,
    top_k=50,
    repetition_penalty=1.3
)

def build_prompt(label):
    examples = random.choices(collected_text, k = 5)

    templates = {
        "positive": (f"Write a short positive customer review or social media post about "
                    f"{COMPANY}, {COMPANY_DESC}. Output only the text.No additional comments or introduction."
                    f"here some stylistic examples with various sentiments{examples}"),

        "negative": (f"Write a short negative complaint or post about "
                    f"{COMPANY}, {COMPANY_DESC}. Output only the text.No additional comments or introduction."
                    f"here some stylistic examples with various sentiments{examples}"),

        "neutral":  (f"Write a short neutral news update or factual statement about" 
                    f"{COMPANY}, {COMPANY_DESC}. Output only the text.No additional comments or introduction."
                    f"here some stylistic examples with various sentiments{examples}"),
    }
    return templates[label]


from tqdm.auto import tqdm

def generate_data(label, n, batch_size=20):
    data = []
    with tqdm(total=n, desc=label) as pbar:
        for _ in range(0, n, batch_size):
            current_batch = min(batch_size, n - len(data))
            prompt = build_prompt(label)
            outputs = generator(
                [[{"role": "user", "content": prompt}]] * current_batch,
            )
            for out in outputs:
                text = out[0]["generated_text"][-1]["content"].strip()
                if text:
                    data.append({"text": text, "label": LABEL_MAP[label]})
            pbar.update(current_batch)
    return data

gen_data = []
for label in LABEL_MAP:
    print(f"Generating {N_PER_CLASS} {label}...")
    gen_data.extend(generate_data(label, N_PER_CLASS))

random.shuffle(gen_data)
print(f"Total: {len(gen_data)}")

print(random.choices(gen_data, k=10))


del generator
torch.cuda.empty_cache()


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length', 'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Generating 300 negative...


negative:   0%|          | 0/300 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
train_val_data = load_dataset("tweet_eval", "sentiment", split="train")
real_test = load_dataset("tweet_eval", "sentiment", split="test")



In [ ]:
if MODEL_EXISTS:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, add_prefix_space=True)
else:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, add_prefix_space=True)

def aspect_tokenize(aspect):
    def _tokenize(batch):
        return tokenizer(batch["text"],[aspect] * len(batch["text"]), max_length=128, truncation=True)
    return _tokenize
def aspect_tokenize(aspect):
    def _tokenize(batch):
        return tokenizer(batch["text"],[aspect] * len(batch["text"]), max_length=128, truncation=True)
    return _tokenize

split = train_val_data.train_test_split(test_size=0.2, seed= 0)
train_data, val_data = split["train"], split["test"]
gen_dataset = Dataset.from_list(gen_data)


SYNTH_RATIO = 0.3  
n_synth = len(gen_dataset)
n_real = int(n_synth * (1 - SYNTH_RATIO) / SYNTH_RATIO)
n_per_label = n_real // 3

sampled_indices = []
for label in [0, 1, 2]:
    label_indices = [i for i, l in enumerate(train_data["label"]) if l == label]
    sampled_indices.extend(random.choices(label_indices, k=n_per_label))

train_data_sampled = train_data.select(sampled_indices).cast_column("label", Value("int64"))

gen_dataset = gen_dataset.map(aspect_tokenize(ASPECT), batched=True)
train_data_sampled = train_data_sampled.map(aspect_tokenize(GENERIC_ASPECT), batched=True)
val_data = val_data.map(aspect_tokenize(GENERIC_ASPECT), batched=True)
real_test = real_test.map(aspect_tokenize(GENERIC_ASPECT), batched=True)

combined_train = concatenate_datasets([gen_dataset, train_data_sampled])


collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f"Train: {len(combined_train)} | Real test: {len(real_test)}")


In [ ]:
class MLflowCallback(TrainerCallback):
    def on_evaluate(self,args, state,control, metrics= None, **kwargs):
        if metrics:
            mlflow.log_metrics(
                {k: v for k, v in metrics.items() if isinstance(v, (int, float))},
                step=int(state.epoch)
            )


In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {"f1": f1_score(labels, preds, average="weighted"),
            "accuracy": accuracy_score(labels,preds),
            "recall": recall_score(labels,preds, average="weighted")}

if MODEL_EXISTS:
    current = AutoModelForSequenceClassification.from_pretrained(MODEL_REPO)
else:
    current = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL)
    
baseline_metrics = Trainer(
    model=current,
    eval_dataset=real_test,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    data_collator=collator,
).evaluate()
baseline_f1 = baseline_metrics["eval_f1"]
baseline_accuracy = baseline_metrics["eval_accuracy"]
baseline_recall = baseline_metrics["eval_recall"]
print(f"Baseline F1 : {baseline_f1:.4f} | Baseline Accuracy : {baseline_accuracy:.4f} | Baseline Recall : {baseline_recall:.4f}")


del current
torch.cuda.empty_cache()


In [ ]:
if MODEL_EXISTS:
    new_model = AutoModelForSequenceClassification.from_pretrained(MODEL_REPO, num_labels=3)
else:
    new_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=3)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16, #rank
    lora_alpha = 32, #scaling
    lora_dropout = 0.1,
    target_modules = ["query", "value"],
)

new_model = get_peft_model(new_model, lora_config)

trainer = Trainer(
    model=new_model,
    args=TrainingArguments(
        output_dir="./results",
        hub_model_id=MODEL_REPO,
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        num_train_epochs=5,
        weight_decay=0.01,
    ),
    train_dataset=combined_train,
    eval_dataset=val_data,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    data_collator=collator,
    callbacks=[MLflowCallback()],
)



In [ ]:
run_timestamp = datetime.now().isoformat()

with mlflow.start_run(run_name = f"v{run_timestamp}"):

    trainer.train()
    new_metrics = trainer.evaluate(eval_dataset= real_test)
    
    new_f1 = new_metrics["eval_f1"]
    new_accuracy = new_metrics["eval_accuracy"]
    new_recall = new_metrics["eval_recall"]
    print(f"New F1: {new_f1:.4f} | New Accuracy : {new_accuracy:.4f} | New Recall : {new_recall:.4f}")
        
    quality_control = new_f1 > baseline_f1

    print(f"Ready for service: {quality_control}  ({baseline_f1:.4f} → {new_f1:.4f})")

    api = HfApi(token=HF_TOKEN)

    payload = {
        "timestamp":            datetime.now().isoformat(),
        "baseline_f1":          round(baseline_f1, 4),
        "baseline_accuracy":    round(baseline_accuracy,4),
        "baseline_recall":      round(baseline_recall,4),
        "new_f1":               round(new_f1, 4),
        "new_accuracy":         round(new_accuracy,4),
        "new_recall":           round(new_recall,4),
        "performance":          quality_control,
        "n_samples":            len(combined_train),
    }
    try:
        path = hf_hub_download(repo_id=METRICS_REPO, filename="history.json", repo_type="dataset")
        history = json.load(open(path))
    except:
        history = []

    history.append(payload)

    api.upload_file(
    path_or_fileobj=json.dumps(history).encode(),
    path_in_repo="history.json",
    repo_id=METRICS_REPO,
    repo_type="dataset",
    commit_message=f"run {datetime.now().strftime('%Y-%m-%d %H:%M')}"
    )
    mlflow.log_metric("baseline_f1", baseline_f1)
    mlflow.log_metric("new_f1", new_f1)
    mlflow.log_metric("new_accuracy", new_accuracy)
    mlflow.log_metric("new_recall", new_recall)
    
    mlflow.set_tag("deployed_to_production", str(quality_control))

    if quality_control:
        trainer.model = trainer.model.merge_and_unload()
        commit_info = trainer.push_to_hub(
            commit_message=f"retrain: F1 {baseline_f1:.3f}→{new_f1:.3f}"
        )

        api.create_tag(
            repo_id=MODEL_REPO, 
            tag=f"v{run_timestamp}", 
            tag_message=f"F1={new_f1:.3f}",
            repo_type="model"
            )
        mlflow.set_tag("hf_hub_commit", commit_info.oid)
        mlflow.set_tag("hf_hub_tag", f"v{run_timestamp}")
        api.restart_space(repo_id=f"{HF_USER}/{SPACE_NAME}")


In [ ]:

try:
    existing = load_dataset(SYNTH_DATA_REPO)
    gen_dataset = concatenate_datasets([existing, gen_dataset])
except Exception:
    create_repo(SYNTH_DATA_REPO, repo_type="dataset", exist_ok=True)

gen_dataset.push_to_hub(SYNTH_DATA_REPO)




: 